In [1]:
import pandas as pd
### Load the dataset
data=pd.read_csv('dataset/SMSSpamCollection',sep='\t',header=None)
### Rename the columns 
data.columns=['label','message']
### Map the labels to binary values
data['label']=data['label'].map({'ham':0,'spam':1})
data.to_csv("dataset/raw_data.csv", index=False)
print("\nraw_data.csv saved successfully!")




raw_data.csv saved successfully!


#### **Initialize Git & DVC**

In [2]:
pip install dvc


  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Preparing metadata (setup.py) ... done
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.41.5-cp312-cp312-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (20 kB)
  Using cached propcache-0.4.1-cp312-cp312-macosx_11_0_arm64.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 6.5 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 

In [3]:
#### Initialize git 
!git init

Initialized empty Git repository in /Users/nisith/Desktop/AppML/AppliedMachineLearning/Assignment-02/.git/


In [5]:
#### initialize DVC
!dvc init -f 

Initialized DVC repository.

You can now commit the changes to git.

+---------------------------------------------------------------------+
|                                                                     |
|        DVC has enabled anonymous aggregate usage analytics.         |
|     Read the analytics documentation (and how to opt-out) here:     |
|             <https://dvc.org/doc/user-guide/analytics>              |
|                                                                     |
+---------------------------------------------------------------------+

What's next?
------------
- Check out the documentation: <https://dvc.org/doc>
- Get help and share ideas: <https://dvc.org/chat>
- Star us on GitHub: <https://github.com/treeverse/dvc>


#### **STEP-1: Create Train , Validation, Test Split (Version-1: random_state=42)**

In [6]:
from sklearn.model_selection import train_test_split
import os 
### First split: Train+Val and test
train_val, test = train_test_split(
    data,
    test_size=0.15,
    stratify=data["label"],
    random_state=42
)
### Second split: train and validation
train, val = train_test_split(
    train_val,
    test_size=0.1765,  # makes final 15%
    stratify=train_val["label"],
    random_state=42
)
train.to_csv("dataset/train.csv", index=False)
val.to_csv("dataset/validation.csv", index=False)
test.to_csv("dataset/test.csv", index=False)

print("Train shape:", train.shape)
print("Validation shape:", val.shape)
print("Test shape:", test.shape)

Train shape: (3900, 2)
Validation shape: (836, 2)
Test shape: (836, 2)


#### **PRINT DISTRIBUTION (VERSION-1)**

In [7]:
print("VERSION 1: Distribution \n")
for file in ["train.csv", "validation.csv", "test.csv"]:
    df = pd.read_csv(os.path.join("dataset", file))
    print(f"{file} distribution:")
    print(df["label"].value_counts(normalize=True))
    print("\n")

VERSION 1: Distribution 

train.csv distribution:
label
0    0.865897
1    0.134103
Name: proportion, dtype: float64


validation.csv distribution:
label
0    0.866029
1    0.133971
Name: proportion, dtype: float64


test.csv distribution:
label
0    0.866029
1    0.133971
Name: proportion, dtype: float64




#### **TRACK WITH DVC**

In [8]:
!dvc add dataset/raw_data.csv
!dvc add dataset/train.csv
!dvc add dataset/validation.csv
!dvc add dataset/test.csv

 ⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/Users/nisith/Desktop/AppML/AppliedMachineLearning/Assig
                                                                                
!
  0%|          |Adding dataset/raw_data.csv to cache  0/1 [00:00<?,     ?file/s]
                                                                                
!
  0%|          |Checking out /Users/nisith/Desktop/App0/1 [00:00<?,    ?files/s]
100% Adding...|████████████████████████████████████████|1/1 [00:00, 80.76file/s]

To track the changes with git, run:

	git add dataset/raw_data.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true
 ⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
  0% Ch

#### **Commit DVC tracking**

In [9]:
!git add dataset/*.dvc .gitignore
!git commit -m "Tracking dataset with DVC"

fatal: pathspec '.gitignore' did not match any files
[main (root-commit) efc7e13] Tracking dataset with DVC
 Committer: Nisith Ranjan Hazra <nisith@Nisiths-MacBook-Air.local>
Your name and email address were configured automatically based
on your username and hostname. Please check that they are accurate.
You can suppress this message by setting them explicitly. Run the
following command and follow the instructions in your editor to edit
your configuration file:

    git config --global --edit

After doing this, you may fix the identity used for this commit with:

    git commit --amend --reset-author

 3 files changed, 6 insertions(+)
 create mode 100644 .dvc/.gitignore
 create mode 100644 .dvc/config
 create mode 100644 .dvcignore


#### **Push Data to DVC storage**

In [10]:
!mkdir dvc_storage
!dvc remote add -d local_remote dvc_storage
!git add .dvc/config
!git commit -m "Configure DVC local remote"
!dvc push


mkdir: dvc_storage: File exists
Setting 'local_remote' as a default remote.
[main 39c32a9] Configure DVC local remote
 Committer: Nisith Ranjan Hazra <nisith@Nisiths-MacBook-Air.local>
Your name and email address were configured automatically based
on your username and hostname. Please check that they are accurate.
You can suppress this message by setting them explicitly. Run the
following command and follow the instructions in your editor to edit
your configuration file:

    git config --global --edit

After doing this, you may fix the identity used for this commit with:

    git commit --amend --reset-author

 1 file changed, 4 insertions(+)
Pushing
!
  0% Checking cache in '/Users/nisith/Desktop/AppML/AppliedMachineLearning/Assig
                                                                                
!
  0% Checking cache in '/Users/nisith/Desktop/AppML/AppliedMachineLearning/Assig
                                                                                
!
  0%|    

#### **Version 2 (Seed 123)**

In [11]:
from sklearn.model_selection import train_test_split
import os 
### First split: Train+Val and test
train_val, test = train_test_split(
    data,
    test_size=0.15,
    stratify=data["label"],
    random_state=12
)
### Second split: train and validation
train, val = train_test_split(
    train_val,
    test_size=0.1765,  # makes final 15%
    stratify=train_val["label"],
    random_state=42
)
train.to_csv("dataset/train.csv", index=False)
val.to_csv("dataset/validation.csv", index=False)
test.to_csv("dataset/test.csv", index=False)

print("Train shape:", train.shape)
print("Validation shape:", val.shape)
print("Test shape:", test.shape)

Train shape: (3900, 2)
Validation shape: (836, 2)
Test shape: (836, 2)


In [12]:
print("VERSION 1: Distribution \n")
for file in ["train.csv", "validation.csv", "test.csv"]:
    df = pd.read_csv(os.path.join("dataset", file))
    print(f"{file} distribution:")
    print(df["label"].value_counts(normalize=True))
    print("\n")

VERSION 1: Distribution 

train.csv distribution:
label
0    0.865897
1    0.134103
Name: proportion, dtype: float64


validation.csv distribution:
label
0    0.866029
1    0.133971
Name: proportion, dtype: float64


test.csv distribution:
label
0    0.866029
1    0.133971
Name: proportion, dtype: float64




#### **Track updated Version**

In [13]:
!dvc add dataset/raw_data.csv
!dvc add dataset/train.csv
!dvc add dataset/validation.csv
!dvc add dataset/test.csv

 ⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/Users/nisith/Desktop/AppML/AppliedMachineLearning/Assig
                                                                                
!
  0%|          |Checking out /Users/nisith/Desktop/App0/1 [00:00<?,    ?files/s]
100% Adding...|████████████████████████████████████████|1/1 [00:00, 91.89file/s]

To track the changes with git, run:

	git add dataset/raw_data.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true
 ⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/Users/nisith/Desktop/AppML/AppliedMachineLearning/Assig
                                                                                
!
  0%|  

In [14]:
!git add dataset/train.csv.dvc
!git add dataset/validation.csv.dvc
!git add dataset/test.csv.dvc

In [15]:
!git commit -m "Version 2: Updated the random state for train-test split"

[main c8dd94c] Version 2: Updated the random state for train-test split
 Committer: Nisith Ranjan Hazra <nisith@Nisiths-MacBook-Air.local>
Your name and email address were configured automatically based
on your username and hostname. Please check that they are accurate.
You can suppress this message by setting them explicitly. Run the
following command and follow the instructions in your editor to edit
your configuration file:

    git config --global --edit

After doing this, you may fix the identity used for this commit with:

    git commit --amend --reset-author

 3 files changed, 15 insertions(+)
 create mode 100644 dataset/test.csv.dvc
 create mode 100644 dataset/train.csv.dvc
 create mode 100644 dataset/validation.csv.dvc


#### **Checkout the OLD Version**

In [16]:
!git log --oneline

c8dd94c (HEAD -> main) Version 2: Updated the random state for train-test split
39c32a9 Configure DVC local remote
efc7e13 Tracking dataset with DVC


In [17]:
!git log 

commit c8dd94c438d4f25901396902bbf67297cdd4844e (HEAD -> main)
Author: Nisith Ranjan Hazra <nisith@Nisiths-MacBook-Air.local>
Date:   Wed Mar 4 00:08:49 2026 +0530

    Version 2: Updated the random state for train-test split

commit 39c32a974ff990d7f0ab1fba8f8aa817203f286d
Author: Nisith Ranjan Hazra <nisith@Nisiths-MacBook-Air.local>
Date:   Wed Mar 4 00:08:22 2026 +0530

    Configure DVC local remote

commit efc7e13c3d8502c1c9aaaf25b60c6c88d9262d3e
Author: Nisith Ranjan Hazra <nisith@Nisiths-MacBook-Air.local>
Date:   Wed Mar 4 00:08:17 2026 +0530

    Tracking dataset with DVC


In [18]:
## Reload the files nd check the distribution again
train = pd.read_csv("dataset/train.csv")
val = pd.read_csv("dataset/validation.csv")     
print("VERSION 2: Distribution \n")
for file in ["train.csv", "validation.csv", "test.csv"]:
    df = pd.read_csv(os.path.join("dataset", file))
    print(f"{file} distribution:")
    print(df["label"].value_counts(normalize=True))
    print("\n")

VERSION 2: Distribution 

train.csv distribution:
label
0    0.865897
1    0.134103
Name: proportion, dtype: float64


validation.csv distribution:
label
0    0.866029
1    0.133971
Name: proportion, dtype: float64


test.csv distribution:
label
0    0.866029
1    0.133971
Name: proportion, dtype: float64


